# 5.1.层和块

之前首次介绍神经网络时，我们关注的是具有单一输出的线性模型。在这里，整个模型只有一个输出。注意，单个神经网络（1）接受一些输入；（2）生成相应的标量输出；（3）具有一组相关 *参数*（parameters），更新这些参数可以优化某目标函数。  

然后，当考虑具有多个输出的网络时，我们利用矢量化算法来描述整层神经元。像单个神经元一样，层（1）接受一组输入，（2）生成相应的输出，（3）由一组可调整参数描述。当我们使用softmax回归时，一个单层本身就是模型。然而，即使我们随后引入了多层感知机，我们仍然可以认为该模型保留了上面所说的基本架构。  

对于多层感知机而言，整个模型及其组成层都是这种架构。整个模型接受原始输入（特征），生成输出（预测），并包含一些参数（所有组成层的参数集合）。同样，每个单独的层接收输入（由前一层提供），生成输出（到下一层的输入），并且具有一组可调参数，这些参数根据从下一层反向传播的信号进行更新。  

事实证明，研究讨论“比单个层大”但“比整个模型小”的组件更有价值。例如，在计算机视觉中广泛流行的ResNet-152架构就有数百层，这些层是由*层组*（groups of layers）的重复模式组成。这个ResNet架构赢得了2015年ImageNet和COCO计算机视觉比赛的识别和检测任务 :[He.Zhang.Ren.ea.2016](https://zh.d2l.ai/chapter_references/zreferences.html#id60)。目前ResNet架构仍然是许多视觉任务的首选架构。在其他的领域，如自然语言处理和语音，
层组以各种重复模式排列的类似架构现在也是普遍存在。  

为了实现这些复杂的网络，我们引入了神经网络*块*的概念。*块*（block）可以描述单个层、由多个层组成的组件或整个模型本身。使用块进行抽象的一个好处是可以将一些块组合成更大的组件，这一过程通常是递归的，如下图所示。通过定义代码来按需生成任意复杂度的块，我们可以通过简洁的代码实现复杂的神经网络。  

<div style="border: solid 12px #ffffff; background-color: #ffffff; width: 100%; height: 180px; display: flex; align-items: center; justify-content: center;">
  <img src="./images/blocks.svg" alt="多个层被组合成块，形成更大的模型" style="width: 100%; height: 100%; object-fit: contain;">
</div>

---

## 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"

import pypto
import torch
import torch_npu
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

---

从编程的角度来看，块由*类*（class）表示。它的任何子类都必须定义一个将其输入转换为输出的前向传播函数，并且必须存储任何必需的参数。注意，有些块不需要任何参数。最后，为了计算梯度，块必须具有反向传播函数。在定义我们自己的块时，由于自动微分（在 [02.05_calculus](../02_pypto_preliminaries/02.05_calculus.ipynb) 中引入）提供了一些后端实现，我们只需要考虑前向传播函数和必需的参数。  

在前面的章节中，我们将 `Linear` 层拆分为独立的 `Matmul` 和 `BiasAdd` 两个 kernel。这种模块化的写法有助于清晰地理解每一步操作的作用，也方便算子的独立复用。在此基础上，本节我们尝试一种更紧凑的实现方式：将矩阵乘法与偏置加法合并为单一 kernel，并配合动态 tile shapes 进行优化。这样可以减少 kernel 调用开销，也更贴近实际部署中的写法。

In [9]:
# ========== 分块策略选择 ==========
def linear_tile_shapes(in_features, out_features):
    if max(in_features, out_features) <= 64:
        return [16, 16], [32, 64], [16, 16] 
    return [32, 32], [64, 64], [64, 64]

def relu_tile_shapes(features):
    if features <= 64:
        return 16, 16
    return 32, 32

# ========== kernel 定义 ==========
def get_pypto_linear_kernel(in_features, out_features):
    m_tile, k_tile, n_tile = linear_tile_shapes(in_features, out_features)

    @pypto.frontend.jit
    def linear_kernel(
            x: pypto.Tensor([], pypto.DT_FP32),
            weight: pypto.Tensor([], pypto.DT_FP32),
            bias: pypto.Tensor([], pypto.DT_FP32),
            out: pypto.Tensor([], pypto.DT_FP32),
        ):
            pypto.set_cube_tile_shapes(m_tile, k_tile, n_tile)
            h = pypto.matmul(x, weight, pypto.DT_FP32, b_trans=True)
            pypto.set_vec_tile_shapes(m_tile[0], n_tile[0])
            out[:] = pypto.add(h, bias)

    return linear_kernel

def get_pypto_relu_kernel(features):
    tile_b, tile_f = relu_tile_shapes(features)

    @pypto.frontend.jit
    def relu_kernel(
        x: pypto.Tensor([], pypto.DT_FP32),
        out: pypto.Tensor([], pypto.DT_FP32),
    ):
        pypto.set_vec_tile_shapes(tile_b, tile_f)
        out[:] = pypto.relu(x)
    return relu_kernel

# ========== 自动微分包装 ==========
class PyPTOLinearFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, X, weight, bias, kernel):
        ctx.save_for_backward(X, weight)
        out = torch.zeros(X.shape[0], weight.shape[0], device=X.device, dtype=X.dtype)
        kernel(X, weight, bias, out)
        return out
    
    @staticmethod
    def backward(ctx, grad_output):
        X, weight = ctx.saved_tensors
        grad_bias = grad_output.sum(dim=0) if grad_output.shape[1] > 0 else None
        grad_weight = grad_output.t() @ X
        grad_X = grad_output @ weight
        return grad_X, grad_weight, grad_bias, None

class PyPTOReLUFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, X, kernel):
        ctx.save_for_backward(X)
        out = torch.empty_like(X)
        kernel(X, out)
        return out
    
    @staticmethod
    def backward(ctx, grad_output):
        (X,) = ctx.saved_tensors
        return grad_output * (X > 0).float(), None

# ========== nn.Module 包装 ==========
class PyPTOLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.device = "npu:0"
        self.weight = nn.Parameter(torch.empty((out_features, in_features), dtype=torch.float32, device=self.device))
        self.bias = nn.Parameter(torch.empty(out_features, dtype=torch.float32, device=self.device))
        self.reset_parameters()
        self._kernel = get_pypto_linear_kernel(in_features, out_features)

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        fan_in = self.weight.shape[1]
        bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, X):
        if X.shape[-1] != self.in_features:
            raise ValueError(f"期望输入维度 {self.in_features}, 得到 {X.shape[-1]}")
        if X.dtype != self.weight.dtype:
            raise TypeError(f"输入类型 {X.dtype} 必须与权重类型 {self.weight.dtype} 一致")

        # 处理多维输入
        leading_shape = X.shape[:-1]  # X.shape[:-1]：取形状的除最后一个之外的所有维度
        X_2d = X.reshape(-1, self.in_features).contiguous()  # -1 表示"自动计算这个维度的大小"
        out_2d = PyPTOLinearFunction.apply(X_2d, self.weight, self.bias, self._kernel)
        
        return out_2d.reshape(*leading_shape, self.out_features)

class PyPTOReLU(nn.Module):
    def __init__(self):
        super().__init__()
        self._kernels = {}

    def forward(self, X):
        features = X.shape[-1]
        if features not in self._kernels:
            self._kernels[features] = get_pypto_relu_kernel(features)
        leading_shape = X.shape[:-1]
        X_2d = X.reshape(-1, features).contiguous()
        out_2d = PyPTOReLUFunction.apply(X_2d, self._kernels[features])
        return out_2d.reshape(*leading_shape, features)

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击展开 / 折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;">由于 PyPTO kernel 目前只支持二维矩阵运算，遇到多维输入时，这里先将其展平为二维再交给 kernel 处理。<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">leading_shape = X.shape[:-1]</code> 记录输入的前导维度，<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">X.reshape(-1, self.in_features).contiguous()</code> 将输入展平为 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">[batch_flat, in_features]</code> 的二维矩阵——其中 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">-1</code> 自动推断批次维度，<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">contiguous()</code> 保证内存布局连续。kernel 计算完成后，<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">reshape(*leading_shape, self.out_features)</code> 再将输出恢复为原始的前导形状。这样处理后，<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">PyPTOLinear</code> 就可以适配任意维度的输入，在使用方式上与 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">nn.Linear</code> 保持一致。</li>
    </ul>
  </div>
</details>

在构造自定义块之前，**我们先回顾一下多层感知机**（ [04.03_mlp_concise](../04_multilayer_perceptrons/04.03_mlp_concise.ipynb) ）的代码。下面的代码生成一个网络，其中包含一个具有256个单元和ReLU激活函数的全连接隐藏层，然后是一个具有10个隐藏单元且不带激活函数的全连接输出层。

In [11]:
net = nn.Sequential(PyPTOLinear(20, 256), PyPTOReLU(), PyPTOLinear(256, 10))

X = torch.rand(2, 20).npu()
net(X)

tensor([[-0.0044, -0.1341,  0.2581,  0.0840, -0.1567,  0.3070, -0.2483,  0.2850,
         -0.2858,  0.2783],
        [-0.0956,  0.0265,  0.2425,  0.0155, -0.2475,  0.1737, -0.1162,  0.1190,
         -0.3000,  0.2590]], device='npu:0', grad_fn=<ViewBackward0>)

在这个例子中，我们通过实例化`nn.Sequential`来构建我们的模型，层的执行顺序是作为参数传递的。简而言之，**`nn.Sequential`定义了一种特殊的`Module`**，即在PyTorch中表示一个块的类，它维护了一个由`Module`组成的有序列表。注意，两个全连接层都是`PyPTOLinear`类的实例，`PyPTOLinear`类本身就是`Module`的子类。另外，到目前为止，我们一直在通过`net(X)`调用我们的模型来获得模型的输出。这实际上是`net.__call__(X)`的简写。这个前向传播函数非常简单：它将列表中的每个块连接在一起，将每个块的输出作为下一个块的输入。

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
  <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-top: none; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">tensor([[ 0.0343,  0.0264,  0.2505, -0.0243,  0.0945,  0.0012, -0.0141,  0.0666,
         -0.0547, -0.0667],
        [ 0.0772, -0.0274,  0.2638, -0.0191,  0.0394, -0.0324,  0.0102,  0.0707,
         -0.1481, -0.1031]], grad_fn=&lt;AddmmBackward0&gt;)</pre>
  在这个例子中，我们通过实例化<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">nn.Sequential</code>来构建我们的模型，层的执行顺序是作为参数传递的。简而言之，<strong><code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">nn.Sequential</code>定义了一种特殊的<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">Module</code></strong>，即在PyTorch中表示一个块的类，它维护了一个由<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">Module</code>组成的有序列表。注意，两个全连接层都是<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">Linear</code>类的实例，<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">Linear</code>类本身就是<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">Module</code>的子类。另外，到目前为止，我们一直在通过<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">net(X)</code>调用我们的模型来获得模型的输出。这实际上是<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">net.__call__(X)</code>的简写。这个前向传播函数非常简单：它将列表中的每个块连接在一起，将每个块的输出作为下一个块的输入。
  </div>
</details>

---

## 5.1.1.自定义块

要想直观地了解块是如何工作的，最简单的方法就是自己实现一个。在实现我们自定义块之前，我们简要总结一下每个块必须提供的基本功能。


1. 将输入数据作为其前向传播函数的参数。
1. 通过前向传播函数来生成输出。请注意，输出的形状可能与输入的形状不同。例如，我们上面模型中的第一个全连接的层接收一个20维的输入，但是返回一个维度为256的输出。
1. 计算其输出关于输入的梯度，可通过其反向传播函数进行访问。在 PyTorch 中通常这是自动发生的。
1. 存储和访问前向传播计算所需的参数。
1. 根据需要初始化模型参数。


在下面的代码片段中，我们从零开始编写一个块。它包含一个多层感知机，其具有256个隐藏单元的隐藏层和一个10维输出层。注意，下面的`MLP`类继承了表示块的类。我们的实现只需要提供我们自己的构造函数（Python中的`__init__`函数）和前向传播函数。

In [6]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self, in_features=20, hidden_features=256, out_features=10):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = PyPTOLinear(in_features, hidden_features)  # 隐藏层
        self.relu = PyPTOReLU()  # 激活函数
        self.out = PyPTOLinear(hidden_features, out_features)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用 PyPTOReLU 模块。
        return self.out(self.relu(self.hidden(X)))

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">class MLP(nn.Module):
    <span style="color:#067d17;"># 用模型参数声明层。这里，我们声明两个全连接的层</span>
    def __init__(self):
        <span style="color:#067d17;"># 调用MLP的父类Module的构造函数来执行必要的初始化。</span>
        <span style="color:#067d17;"># 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）</span>
        super().__init__()
        self.hidden = nn.Linear(20, 256)  <span style="color:#067d17;"># 隐藏层</span>
        self.out = nn.Linear(256, 10)  <span style="color:#067d17;"># 输出层</span>
    <span style="color:#067d17;"># 定义模型的前向传播，即如何根据输入X返回所需的模型输出</span>
    def forward(self, X):
        <span style="color:#067d17;"># 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。</span>
        return self.out(F.relu(self.hidden(X)))</pre>
  </div>
</details>

我们首先看一下前向传播函数，它以`X`作为输入，计算带有激活函数的隐藏表示，并输出其未规范化的输出值。在这个`MLP`实现中，两个层都是实例变量。要了解这为什么是合理的，可以想象实例化两个多层感知机（`net1`和`net2`），并根据不同的数据对它们进行训练。当然，我们希望它们学到两种不同的模型。

接着我们**实例化多层感知机的层，然后在每次调用前向传播函数时调用这些层**。注意一些关键细节：首先，我们定制的`__init__`函数通过`super().__init__()`调用父类的`__init__`函数，省去了重复编写模版代码的痛苦。然后，我们实例化两个全连接层，分别为`self.hidden`和`self.out`。注意，除非我们实现一个新的运算符，否则我们不必担心反向传播函数或参数初始化，系统将自动生成这些。

我们来试一下这个函数：

In [7]:
net = MLP()
net(X)

tensor([[ 0.0188, -0.1973, -0.0806, -0.0195, -0.1542,  0.2600,  0.0715,  0.0269,
         -0.2240, -0.1062],
        [ 0.0403,  0.0433,  0.0301,  0.0409, -0.1930,  0.2304,  0.0384,  0.1028,
         -0.1863, -0.0804]], device='npu:0', grad_fn=<ViewBackward0>)

块的一个主要优点是它的多功能性。我们可以子类化块以创建层（如全连接层的类）、整个模型（如上面的`MLP`类）或具有中等复杂度的各种组件。我们在接下来的章节中充分利用了这种多功能性，比如在处理卷积神经网络时。

---

## 5.1.2.顺序块

现在我们可以更仔细地看看`Sequential`类是如何工作的，回想一下`Sequential`的设计是为了把其他模块串起来。为了构建我们自己的简化的`MySequential`，我们只需要定义两个关键函数：

1. 一种将块逐个追加到列表中的函数；
1. 一种前向传播函数，用于将输入按追加块的顺序传递给块组成的“链条”。

下面的`MySequential`类提供了与默认`Sequential`类相同的功能。

In [6]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

`__init__`函数将每个模块逐个添加到有序字典`_modules`中。读者可能会好奇为什么每个`Module`都有一个`_modules`属性？以及为什么我们使用它而不是自己定义一个Python列表？简而言之，`_modules`的主要优点是：在模块的参数初始化过程中，系统知道在`_modules`字典中查找需要初始化参数的子块。


当`MySequential`的前向传播函数被调用时，
每个添加的块都按照它们被添加的顺序执行。
现在可以使用我们的`MySequential`类重新实现多层感知机。


In [9]:
net = MySequential(PyPTOLinear(20, 256), PyPTOReLU(), PyPTOLinear(256, 10))
net(X)

tensor([[-0.0091, -0.0925,  0.1032,  0.1525, -0.1581, -0.0006,  0.2348, -0.1079,
          0.0222,  0.0004],
        [-0.0853,  0.0394,  0.0801,  0.0940, -0.1737, -0.0523,  0.2074, -0.0148,
          0.1960,  0.0781]], device='npu:0', grad_fn=<ViewBackward0>)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)</pre>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-top: none; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">
tensor([[ 2.2759e-01, -4.7003e-02,  4.2846e-01, -1.2546e-01,  1.5296e-01,
          1.8972e-01,  9.7048e-02,  4.5479e-04, -3.7986e-02,  6.4842e-02],
        [ 2.7825e-01, -9.7517e-02,  4.8541e-01, -2.4519e-01, -8.4580e-02,
          2.8538e-01,  3.6861e-02,  2.9411e-02, -1.0612e-01,  1.2620e-01]],
       grad_fn=&lt;AddmmBackward0&gt;)
</pre>
  </div>
</details>

请注意，`MySequential`的用法与之前为`Sequential`类编写的代码相同。

---

## 5.1.3.在前向传播函数中执行代码

`Sequential`类使模型构造变得简单，允许我们组合新的架构，而不必定义自己的类。然而，并不是所有的架构都是简单的顺序架构。当需要更强的灵活性时，我们需要定义自己的块。例如，我们可能希望在前向传播函数中执行Python的控制流。此外，我们可能希望执行任意的数学运算，而不是简单地依赖预定义的神经网络层。

到目前为止，我们网络中的所有操作都对网络的激活值及网络的参数起作用。然而，有时我们可能希望合并既不是上一层的结果也不是可更新参数的项，我们称之为*常数参数*（constant parameter）。例如，我们需要一个计算函数$f(\mathbf{x},\mathbf{w}) = c \cdot \mathbf{w}^\top \mathbf{x}$的层，其中$\mathbf{x}$是输入，$\mathbf{w}$是参数，$c$是某个在优化过程中没有更新的指定常量。因此我们实现了一个`FixedHiddenMLP`类。  

In [10]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False, device="npu:0")
        self.linear = PyPTOLinear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        <span style="color:#067d17;"># 不计算梯度的随机权重参数</span>
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)
    def forward(self, X):
        X = self.linear(X)
        <span style="color:#067d17;"># 使用创建的常量参数以及relu和mm函数</span>
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        <span style="color:#067d17;"># 复用全连接层。这相当于两个全连接层共享参数</span>
        X = self.linear(X)
        <span style="color:#067d17;"># 控制流</span>
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()</pre>
  </div>
</details>

在这个`FixedHiddenMLP`模型中，我们实现了一个隐藏层，其权重（`self.rand_weight`）在实例化时被随机初始化，之后为常量。这个权重不是一个模型参数，因此它永远不会被反向传播更新。然后，神经网络将这个固定层的输出通过一个全连接层。  

注意，在返回输出之前，模型做了一些不寻常的事情：它运行了一个 while 循环，在$L_1$范数大于$1$的条件下，将输出向量除以$2$，直到它满足条件为止。最后，模型返回了`X`中所有项的和。注意，此操作可能不会常用于在任何实际任务中，我们只展示如何将任意代码集成到神经网络计算的流程中。

In [11]:
net = FixedHiddenMLP()
net(X)

tensor(-0.0758, device='npu:0', grad_fn=<SumBackward0>)

我们可以**混合搭配各种组合块的方法**。在下面的例子中，我们以一些想到的方法嵌套块。


In [12]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(PyPTOLinear(20, 64), PyPTOReLU(),
                                 PyPTOLinear(64, 32), PyPTOReLU())
        self.linear = PyPTOLinear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), PyPTOLinear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(-0.0607, device='npu:0', grad_fn=<SumBackward0>)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)
    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)</pre>
  <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-top: none; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">tensor(0.2183, grad_fn=&lt;SumBackward0&gt;)</pre>
  </div>
</details>

---

## 5.1.4.效率


读者可能会开始担心操作效率的问题。毕竟，我们在一个高性能的深度学习库中进行了大量的字典查找、代码执行和许多其他的Python代码。Python的问题[全局解释器锁](https://wiki.python.org/moin/GlobalInterpreterLock)是众所周知的。在深度学习环境中，我们担心速度极快的GPU可能要等到CPU运行Python代码后才能运行另一个作业。

---

## 5.1.5.小结

* 一个块可以由许多层组成；一个块可以由许多块组成。
* 块可以包含代码。
* 块负责大量的内部处理，包括参数初始化和反向传播。
* 层和块的顺序连接由`Sequential`块处理。

---
## 5.1.6.练习

1. 如果将`MySequential`中存储块的方式更改为Python列表，会出现什么样的问题？
1. 实现一个块，它以两个块为参数，例如`net1`和`net2`，并返回前向传播中两个网络的串联输出。这也被称为平行块。
1. 假设我们想要连接同一网络的多个实例。实现一个函数，该函数生成同一个块的多个实例，并在此基础上构建更大的网络。

详细参考答案见[05.01_reference_answer](./answers/05.01_reference_answer.ipynb)

### 参考答案（PyPTO）

In [ ]:
!cat ./answers/txt/05.01_reference_pypto.txt

### 参考答案（PyTorch）

In [ ]:
!cat ./answers/txt/05.01_reference_pytorch.txt